# Trend Analysis

Time-series analysis across all available snapshots to track long-term trajectory in build performance, codebase metrics, and efficiency trends.

In [ ]:
from __future__ import annotations
import warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore", category=FutureWarning)

SNAPSHOTS_DIR = Path("../../data/snapshots")
CONFIG_DIR = Path("../../data/config")

from buildanalysis.snapshots import SnapshotManager

sm = SnapshotManager(SNAPSHOTS_DIR)
all_snapshots = sm.list_snapshots()

plt.rcParams.update({"figure.figsize": (12, 6), "figure.dpi": 100, "axes.titlesize": 14})
sns.set_theme(style="whitegrid", palette="muted")

print(f"Found {len(all_snapshots)} snapshot(s)")

## 1. Snapshot Timeline

In [ ]:
if len(all_snapshots) > 0:
    timeline_data = []
    for snap in all_snapshots:
        interventions = snap.interventions_applied if hasattr(snap, "interventions_applied") else []
        interventions_str = "; ".join(interventions) if isinstance(interventions, list) else str(interventions)

        timeline_data.append({
            "label": snap.label,
            "date": snap.date,
            "git_ref": snap.git_ref if hasattr(snap, "git_ref") else "N/A",
            "interventions": interventions_str,
        })

    timeline_df = pd.DataFrame(timeline_data)

    print("\nSNAPSHOT TIMELINE")
    print("=" * 120)
    print(timeline_df.to_string(index=False))
else:
    print("No snapshots available for timeline analysis")

In [ ]:
from buildanalysis.comparison import compute_trend_data, compute_module_trends, detect_regressions

if len(all_snapshots) > 1:
    all_snap_pairs = sm.load_all()
    trend_df = compute_trend_data(all_snap_pairs)
    print(f"Trend data loaded: {len(trend_df)} snapshots")
    print(trend_df[["label", "date", "total_build_time_ms", "target_count", "file_count", "total_sloc", "edge_count", "mean_dep_count"]].to_string(index=False))
else:
    trend_df = pd.DataFrame()
    print("Need at least 2 snapshots for trend analysis")

## 2. Build Time Trend

In [ ]:
if len(all_snapshots) > 1:
    print("\nBUILD TIME TREND")
    print("=" * 80)
    print(trend_df[["label", "date", "total_build_time_ms"]].to_string(index=False))

    # Plot
    fig, ax = plt.subplots(figsize=(12, 6))
    x = range(len(trend_df))
    ax.plot(x, trend_df["total_build_time_ms"] / 1000, marker="o", linewidth=2, markersize=8, label="Total Build Time")
    ax.set_xticks(x)
    ax.set_xticklabels(trend_df["label"], rotation=45, ha="right")
    ax.set_ylabel("Total Build Time (seconds)")
    ax.set_title("Build Time Trend Over Snapshots")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Need at least 2 snapshots for trend analysis")

## 3. Codebase Growth

In [ ]:
if len(all_snapshots) > 1:
    print("\nCODEBASE GROWTH")
    print("=" * 80)
    print(trend_df[["label", "total_sloc", "target_count", "file_count"]].to_string(index=False))

    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    x = range(len(trend_df))

    # SLOC
    axes[0].plot(x, trend_df["total_sloc"] / 1e6, marker="o", linewidth=2, markersize=8, color="#1f77b4")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(trend_df["label"], rotation=45, ha="right")
    axes[0].set_ylabel("Total SLOC (millions)")
    axes[0].set_title("Code Volume Trend")
    axes[0].grid(True, alpha=0.3)

    # Target count
    axes[1].plot(x, trend_df["target_count"], marker="s", linewidth=2, markersize=8, color="#ff7f0e")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(trend_df["label"], rotation=45, ha="right")
    axes[1].set_ylabel("Target Count")
    axes[1].set_title("Build Target Count Trend")
    axes[1].grid(True, alpha=0.3)

    # File count
    axes[2].plot(x, trend_df["file_count"], marker="^", linewidth=2, markersize=8, color="#2ca02c")
    axes[2].set_xticks(x)
    axes[2].set_xticklabels(trend_df["label"], rotation=45, ha="right")
    axes[2].set_ylabel("File Count")
    axes[2].set_title("Source File Count Trend")
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("Need at least 2 snapshots for growth analysis")

## 4. Efficiency Trend

In [ ]:
if len(all_snapshots) > 1:
    efficiency_df = trend_df.copy()
    efficiency_df["build_time_per_sloc"] = efficiency_df.apply(
        lambda r: r["total_build_time_ms"] / r["total_sloc"] if r["total_sloc"] > 0 else 0,
        axis=1
    )

    print("\nEFFICIENCY ANALYSIS (Build Time per SLOC)")
    print("=" * 90)
    print(efficiency_df[["label", "date", "build_time_per_sloc", "total_build_time_ms", "total_sloc"]].to_string(index=False))

    # Determine trend
    if len(efficiency_df) > 1:
        first_eff = efficiency_df["build_time_per_sloc"].iloc[0]
        last_eff = efficiency_df["build_time_per_sloc"].iloc[-1]
        delta_eff = last_eff - first_eff
        delta_pct = 100 * delta_eff / first_eff if first_eff > 0 else 0

        if delta_eff < 0:
            print(f"\nIMPROVING: Build efficiency improved by {abs(delta_pct):.1f}%")
        elif delta_eff > 0:
            print(f"\nDEGRADING: Build efficiency degraded by {delta_pct:.1f}%")
        else:
            print(f"\nSTABLE: Build efficiency unchanged")

    # Plot
    fig, ax = plt.subplots(figsize=(12, 6))
    x = range(len(efficiency_df))
    ax.plot(x, efficiency_df["build_time_per_sloc"], marker="o", linewidth=2, markersize=8, color="#d62728")
    ax.set_xticks(x)
    ax.set_xticklabels(efficiency_df["label"], rotation=45, ha="right")
    ax.set_ylabel("Build Time per SLOC (ms/line)")
    ax.set_title("Build Efficiency Trend")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Need at least 2 snapshots for efficiency analysis")

## 5. Dependency Density

In [ ]:
if len(all_snapshots) > 1:
    density_df = trend_df[["label", "date", "edge_count", "target_count", "mean_dep_count"]].copy()

    print("\nDEPENDENCY DENSITY")
    print("=" * 90)
    print(density_df.to_string(index=False))

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    x = range(len(density_df))

    # Edge count
    axes[0].plot(x, density_df["edge_count"], marker="o", linewidth=2, markersize=8, label="Edge Count", color="#1f77b4")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(density_df["label"], rotation=45, ha="right")
    axes[0].set_ylabel("Edge Count")
    axes[0].set_title("Dependency Edge Count Trend")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    # Mean dep count
    axes[1].plot(x, density_df["mean_dep_count"], marker="s", linewidth=2, markersize=8, label="Mean Deps/Target", color="#ff7f0e")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(density_df["label"], rotation=45, ha="right")
    axes[1].set_ylabel("Mean Dependency Count")
    axes[1].set_title("Mean Dependency Count per Target")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    plt.tight_layout()
    plt.show()
else:
    print("Need at least 2 snapshots for density analysis")

## 6. Module Trends

In [ ]:
if len(all_snapshots) > 1:
    modules_path = CONFIG_DIR / "modules.yaml"

    if modules_path.exists():
        from buildanalysis.modules import ModuleConfig

        module_config = ModuleConfig.from_yaml(modules_path)
        module_trend_df = compute_module_trends(all_snap_pairs, module_config)

        print("\nMODULE BUILD TIME TRENDS")
        print("=" * 90)

        if not module_trend_df.empty:
            # Plot top modules
            fig, ax = plt.subplots(figsize=(14, 7))
            x = range(len(trend_df))
            x_labels = list(trend_df["label"])

            top_modules = (
                module_trend_df.groupby("module")["total_build_time_ms"]
                .sum()
                .nlargest(5)
                .index
            )

            for module_name in top_modules:
                module_rows = module_trend_df[module_trend_df["module"] == module_name].sort_values("date")
                times = list(module_rows["total_build_time_ms"] / 1000)
                ax.plot(x[:len(times)], times, marker="o", linewidth=2, markersize=8, label=module_name)

            ax.set_xticks(x)
            ax.set_xticklabels(x_labels, rotation=45, ha="right")
            ax.set_ylabel("Module Build Time (seconds)")
            ax.set_title("Top 5 Modules: Build Time Trend")
            ax.legend(loc="best")
            ax.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()
        else:
            print("No module trend data available.")
    else:
        print("No module config available. Skipping module trends.")
else:
    print("Need at least 2 snapshots for module trend analysis")

## 7. Regression Alerts

In [ ]:
if len(all_snapshots) > 1:
    print("\nREGRESSION DETECTION (10% threshold)")
    print("=" * 90)

    REGRESSION_THRESHOLD = 10.0  # percent

    regressions_df = detect_regressions(trend_df, threshold_pct=REGRESSION_THRESHOLD)

    if not regressions_df.empty:
        print(f"Found {len(regressions_df)} regression(s) > {REGRESSION_THRESHOLD:.0f}%:")
        print(regressions_df.to_string(index=False))
    else:
        print(f"No regressions detected (threshold: {REGRESSION_THRESHOLD:.0f}%)")
else:
    regressions_df = pd.DataFrame()
    print("Need at least 2 snapshots for regression detection")

## 8. Summary

In [ ]:
if len(all_snapshots) > 1:
    print("\n" + "=" * 90)
    print("TREND ANALYSIS SUMMARY")
    print("=" * 90)

    baseline_time = trend_df["total_build_time_ms"].iloc[0]
    latest_time = trend_df["total_build_time_ms"].iloc[-1]
    total_delta_pct = 100 * (latest_time - baseline_time) / baseline_time if baseline_time > 0 else 0

    print(f"\nSnapshot Count: {len(all_snapshots)}")
    print(f"Baseline Build Time (first snapshot): {baseline_time/1000:.1f} seconds")
    print(f"Latest Build Time (last snapshot): {latest_time/1000:.1f} seconds")
    print(f"Overall Change: {total_delta_pct:+.1f}%")

    if total_delta_pct < 0:
        print(f"\nTRAJECTORY: IMPROVING (down {abs(total_delta_pct):.1f}%)")
    elif total_delta_pct > 0:
        print(f"\nTRAJECTORY: DEGRADING (up {total_delta_pct:.1f}%)")
    else:
        print(f"\nTRAJECTORY: STABLE")

    # Metrics summary
    print("\nKey Trends:")
    print(f"  - Codebase size: {trend_df['total_sloc'].iloc[0]/1e6:.2f}M -> {trend_df['total_sloc'].iloc[-1]/1e6:.2f}M SLOC")
    print(f"  - Target count: {trend_df['target_count'].iloc[0]} -> {trend_df['target_count'].iloc[-1]} targets")
    print(f"  - Dependency density: {density_df['mean_dep_count'].iloc[0]:.2f} -> {density_df['mean_dep_count'].iloc[-1]:.2f} deps/target")

    if not regressions_df.empty:
        print(f"\nRegressions detected: {len(regressions_df)}")
    else:
        print(f"\nNo major regressions detected")

    print("\n" + "=" * 90)
else:
    print("Need at least 2 snapshots to generate trend summary")